# Inspector

Inspector is a prototype DESI spectral viewer for use with jupyter notebooks at NERSC.  It is currently in a standalone package for exploratory development, but could be moved into desispec after it matures.

In [ ]:
import os
import inspector

## Loading Spectra

Use `inspector.load_spectra(specfile)` to load a `spectra*.fits` file.  A matching `zbest*.fits` file should be in the same directory; otherwise use the optional `zbestfile` argument to specify the location of a corresponding redrock zbest output file.

In [ ]:
reference_run = '18.6'
specprod = 'mini'

specfile = os.path.join(
    os.environ['DESI_ROOT'], 'datachallenge', 'reference_runs', reference_run,
    'spectro', 'redux', specprod, 'spectra-64', '52', '5261', 'spectra-64-5261.fits')

specfile = '/Users/sbailey/desi/dev/bokeh/data/spectra-64-5261.fits'

sp = inspector.load_spectra(specfile)

At this point you could sub-select the targets before plotting them,
e.g. select just QSOs or just targetids that had previously been identified for expert followup.
For now, we'll just proceed with plotting.

## Plotting

Start the interactive viewer with the `.plot()` command.
* The large window displays a downsampled spectrum with best fit model overlaid.
* Scanning the mouse over that plot shows a full-resolution zoom of that portion of the spectrum, which is particularly useful for scanning narrow emission lines.
* Try the various tools at the top for panning, zooming, selecting.  When the wheel zoom tool is selected (default), using the scroll wheel will zoom in and out.  Point the mouse over the x or y axis before scrolling to zoom on just that axis.
* Click on one of the legend entries to turn that item on/off.
* The buttons at the bottom can be used to navigate between targets:
  * **prev** / **next**: go to the previous or next target without recording visual inspection results
  * the other buttons record visual inspection results and advance to the next target:
    * **flag**: flag this target has having bad data
    * **no**: the fitted redshift is incorrect
    * **maybe**: the fitted redshift might be correct, but not confidently
    * **yes**: the fitted redshift is definitely correct

Known bug: when navigating from one target to the next, the auto-scaling of the y-axis only works after you use the pan tool to shift the location of a spectrum.  After that the auto-scaling starts working for future targets.


In [ ]:
sp.plot()

## Toggle spectral lines

The display of emission and absorption lines can be toggled with the commands below.  ``True`` turns them on and ``False`` turns them off.  You can also call these functions with no arguments to simply flip the existing state.

In [ ]:
sp.emission(True)

In [ ]:
sp.absorption(True)

In [ ]:
sp.emission(False)

In [ ]:
sp.absorption(False)

In [ ]:
sp.emission()
sp.absorption()

## Visual inpsection results

The visual inspection results are stored in a `visual_scan` table which can be inspected and saved.  This table includes the scanner's username so that results from different scanners can be cross checked.  Tables for different spectra files can also be stacked to create truth tables. 

In [ ]:
sp.visual_scan

In [ ]:
sp.visual_scan.write('scan_results.fits', overwrite=True)

## Navigating through targets

You can use the buttons to go the previous/next targets, or use the `.prev()` and `.next()` functions.  e.g. to watch a loop over all the targets, you could do the following (scroll back up to watch the display update)

In [ ]:
import time
sp.ispec = 0
for i in range(sp.nspec):
    sp.next()
    time.sleep(0.5)

## Sub-selecting targets

You can use the `.select(targetids)` function to sub-select which targets to display.  This discards the other
targets, so if you want to get them back you have to reload the spectra from the beginning.

**Known bug**: sub-selecting the targets doesn't auto-update the display until `.next()` or `.prev()` is called.

e.g. to select just targets identified as QSOs by redrock:

In [ ]:
qso_targetids = sp.zbest['TARGETID'][sp.zbest['SPECTYPE'] == 'QSO']
sp.select(qso_targetids)
sp.prev()

And then you can proceed to scan through them with `.next()`

In [ ]:
sp.next()